# **03 - Data Preparation for Modelling**

This notebook prepares the data for training, applying text normalisation, TF-IDF vectorisation, and generates class-imbalance compensated training sets.

**Pipeline**:
1. ML text normalisation (spaCy lemmatisation + stop-word removal)
2. Stratified split (train / val / test)
3. TF-IDF vectorisation (fit on train only)
4. Class-imbalance compensation: MLSMOTE oversampling
5. Export splits and features for downstream training

## **1. Imports & Configuration**

In [1]:
import sys, os, re, time, pickle, warnings
import numpy as np
import pandas as pd
import spacy
from scipy.sparse import vstack, save_npz

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

warnings.filterwarnings('ignore')
sys.path.append('../src')
from utils import (
    OUTPUT_DIR, PROCESSED_ML_DIR, PROCESSED_DL_DIR,
    RANDOM_SEED, TEST_SIZE, VAL_SIZE, ODS_ALL
)
from schema import ProcessedData

os.makedirs(PROCESSED_ML_DIR, exist_ok=True)
os.makedirs(PROCESSED_DL_DIR, exist_ok=True)

nlp = spacy.load('ca_core_news_lg')

print(f'Seed : {RANDOM_SEED}')


Seed : 42


## **2. Load Processed Dataset**

In [2]:
df = pd.read_parquet(os.path.join(OUTPUT_DIR, 'processed_dataset.parquet'))
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

Shape: (19284, 11)
Columns: ['ANH_ID', 'ANU_DATA_PUBLICACIO', 'ANU_NUM_REGISTRE', 'ORG_NOM', 'ANH_TITOL', 'Tipus Anunci', 'PDF', 'ods_list', 'text', 'full_text', 'text_dl']


## **3. ML Text Normalisation (spaCy)**

Applied after `clean_base` from notebook 01. Kept here (not in 01) because the spaCy
pass is heavy (~20 min) and is only needed by the ML pipeline.

In [3]:
def process_ml(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\u00e0-\u00fc\u00b7\s]', ' ', text)
    doc  = nlp(text)
    tokens = [t.lemma_ for t in doc if not t.is_stop and len(t.text) > 2]
    return ' '.join(tokens)

start = time.time()
df[ProcessedData.TEXT_ML] = df[ProcessedData.FULL_TEXT].apply(process_ml)
print(f'ML normalisation: {time.time() - start:.1f} s')
print(f'Empty after normalisation: {(df[ProcessedData.TEXT_ML].str.strip() == "").sum()}')

ML normalisation: 7058.8 s
Empty after normalisation: 0


## **4. Label Binarisation**

In [4]:
mlb = MultiLabelBinarizer(classes=ODS_ALL)
Y   = mlb.fit_transform(df[ProcessedData.ODS_LIST])
ods_cols = mlb.classes_.tolist()

print(f'Label matrix: {Y.shape}')
for ods, cnt in sorted(zip(ods_cols, Y.sum(axis=0)), key=lambda x: -x[1]):
    print(f'  {ods}: {int(cnt):,}')

Label matrix: (19284, 17)
  ODS 11: 9,308
  ODS 8: 8,970
  ODS 3: 3,279
  ODS 10: 2,907
  ODS 9: 2,823
  ODS 16: 2,333
  ODS 4: 2,221
  ODS 17: 2,002
  ODS 13: 1,859
  ODS 1: 1,497
  ODS 7: 887
  ODS 15: 849
  ODS 5: 704
  ODS 6: 664
  ODS 2: 648
  ODS 12: 444
  ODS 14: 39


## **5. Stratified Split**

In [5]:
X_texts = df[ProcessedData.TEXT_ML].values

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_val_idx, test_idx = next(msss.split(X_texts, Y))

val_size_adj = VAL_SIZE / (1 - TEST_SIZE)
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=val_size_adj, random_state=RANDOM_SEED)
train_sub_idx, val_sub_idx = next(msss2.split(train_val_idx, Y[train_val_idx]))

train_idx = train_val_idx[train_sub_idx]
val_idx   = train_val_idx[val_sub_idx]

print(f'Train: {len(train_idx):,}  Val: {len(val_idx):,}  Test: {len(test_idx):,}')

Train: 13,544  Val: 2,877  Test: 2,863


## **6. TF-IDF Vectorisation**

In [6]:
tfidf   = TfidfVectorizer(max_features=5000, sublinear_tf=True, min_df=3, ngram_range=(1, 2))
X_train = tfidf.fit_transform(X_texts[train_idx])
X_val   = tfidf.transform(X_texts[val_idx])
X_test  = tfidf.transform(X_texts[test_idx])

Y_train, Y_val, Y_test = Y[train_idx], Y[val_idx], Y[test_idx]

print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')

with open(os.path.join(PROCESSED_ML_DIR, 'tfidf_model.pkl'), 'wb') as f: pickle.dump(tfidf, f)
with open(os.path.join(PROCESSED_ML_DIR, 'mlb.pkl'),        'wb') as f: pickle.dump(mlb,   f)
print('TF-IDF and MLB saved.')

X_train: (13544, 5000)  X_val: (2877, 5000)  X_test: (2863, 5000)
TF-IDF and MLB saved.


## **7. Class-Imbalance Compensation**

### Strategy A — `class_weight` (recommended)
Each sub-classifier is penalised inversely proportional to label frequency.
Zero additional training cost; works well when imbalance ratio < 100.
For ODS 14 (ratio ~480) the penalty may still be insufficient — see strategy B.

### Strategy B — `oversampling` (MLSMOTE)
Synthetic samples are generated for minority labels in TF-IDF space (SMOTE-style
interpolation between k-NN pairs). Applied only to the training split to prevent leakage.
Recommended if macro-F1 on minority labels remains below 0.20 after strategy A.

In [7]:
from sklearn.utils.class_weight import compute_class_weight

# Strategy A — per-label class weights for OvR sklearn classifiers
# Each binary sub-classifier receives pos/neg weights inversely proportional to frequency.
class_weights = {}
for i, ods in enumerate(ods_cols):
    w = compute_class_weight('balanced', classes=np.array([0, 1]), y=Y_train[:, i])
    class_weights[ods] = {0: float(w[0]), 1: float(w[1])}

print('Per-label class weights (sorted by positive-class weight, highest first):')
for ods, w in sorted(class_weights.items(), key=lambda x: -x[1][1]):
    ratio = w[1] / w[0]
    print(f'  {ods:8s}  pos={w[1]:.2f}  neg={w[0]:.2f}  ratio={ratio:.1f}x')

Per-label class weights (sorted by positive-class weight, highest first):
  ODS 14    pos=250.81  neg=0.50  ratio=500.6x
  ODS 12    pos=21.85  neg=0.51  ratio=42.7x
  ODS 2     pos=14.92  neg=0.52  ratio=28.8x
  ODS 6     pos=14.59  neg=0.52  ratio=28.2x
  ODS 5     pos=13.76  neg=0.52  ratio=26.5x
  ODS 15    pos=11.38  neg=0.52  ratio=21.8x
  ODS 7     pos=10.90  neg=0.52  ratio=20.8x
  ODS 1     pos=6.46  neg=0.54  ratio=11.9x
  ODS 13    pos=5.21  neg=0.55  ratio=9.4x
  ODS 17    pos=4.83  neg=0.56  ratio=8.7x
  ODS 4     pos=4.35  neg=0.56  ratio=7.7x
  ODS 16    pos=4.15  neg=0.57  ratio=7.3x
  ODS 9     pos=3.43  neg=0.59  ratio=5.9x
  ODS 10    pos=3.33  neg=0.59  ratio=5.7x
  ODS 3     pos=2.95  neg=0.60  ratio=4.9x
  ODS 8     pos=1.08  neg=0.93  ratio=1.2x
  ODS 11    pos=1.04  neg=0.96  ratio=1.1x


In [8]:
def mlsmote(X_sp, Y_arr, n_target=300, k=5, seed=42):
    """MLSMOTE for sparse TF-IDF matrices. Synthesises minority-label samples."""
    from sklearn.neighbors import NearestNeighbors
    rng = np.random.default_rng(seed)
    X_syn, Y_syn = [], []
    for li in range(Y_arr.shape[1]):
        pos  = np.where(Y_arr[:, li] == 1)[0]
        if len(pos) == 0 or len(pos) >= n_target:
            continue
        n_gen = n_target - len(pos)
        X_pos = X_sp[pos]
        knn   = NearestNeighbors(n_neighbors=min(k, len(pos)), metric='cosine').fit(X_pos)
        nn    = knn.kneighbors(X_pos, return_distance=False)
        for _ in range(n_gen):
            a   = rng.integers(0, len(pos))
            b   = nn[a, rng.integers(1, nn.shape[1])]
            alpha = rng.uniform()
            X_syn.append(X_pos[a] + alpha * (X_pos[b] - X_pos[a]))
            Y_syn.append((Y_arr[pos[a]] | Y_arr[pos[b]]).astype(np.uint8))
    if not X_syn:
        return X_sp, Y_arr
    X_out = vstack([X_sp] + X_syn)
    Y_out = np.vstack([Y_arr] + Y_syn)
    print(f'  MLSMOTE: +{len(X_syn):,} synthetic samples → {X_out.shape[0]:,} total')
    return X_out, Y_out

In [9]:
print('Generating MLSMOTE augmented training data...')
start = time.time()
X_train_smote, Y_train_smote = mlsmote(X_train, Y_train, n_target=300, seed=RANDOM_SEED)
print(f'Done in {time.time() - start:.1f} s')


Generating MLSMOTE augmented training data...
  MLSMOTE: +273 synthetic samples → 13,817 total
Done in 1.4 s


## **8. Export Splits & Features**


In [10]:
def export_splits(dataframe, y_matrix, indices, folder, text_col):
    for name, idx in zip(['train', 'val', 'test'], indices):
        subset    = dataframe.iloc[idx][[text_col, ProcessedData.ODS_LIST]]
        labels_df = pd.DataFrame(y_matrix[idx], columns=ods_cols, index=subset.index)
        pd.concat([subset, labels_df], axis=1).to_parquet(
            os.path.join(folder, f'split_{name}.parquet'), index=False
        )

# 1. Export Parquet Dataframes (Text & Labels)
export_splits(df, Y, [train_idx, val_idx, test_idx], PROCESSED_ML_DIR, ProcessedData.TEXT_ML)
export_splits(df, Y, [train_idx, val_idx, test_idx], PROCESSED_DL_DIR, ProcessedData.TEXT_DL)

# 2. Export TF-IDF Matrices (Sparse) and Labels (Numpy)
save_npz(os.path.join(PROCESSED_ML_DIR, 'X_train.npz'), X_train)
np.save(os.path.join(PROCESSED_ML_DIR, 'Y_train.npy'), Y_train)

save_npz(os.path.join(PROCESSED_ML_DIR, 'X_train_smote.npz'), X_train_smote)
np.save(os.path.join(PROCESSED_ML_DIR, 'Y_train_smote.npy'), Y_train_smote)

save_npz(os.path.join(PROCESSED_ML_DIR, 'X_val.npz'), X_val)
np.save(os.path.join(PROCESSED_ML_DIR, 'Y_val.npy'), Y_val)

save_npz(os.path.join(PROCESSED_ML_DIR, 'X_test.npz'), X_test)
np.save(os.path.join(PROCESSED_ML_DIR, 'Y_test.npy'), Y_test)

# 3. Export Class Weights (Strategy A)
with open(os.path.join(PROCESSED_ML_DIR, 'class_weights.pkl'), 'wb') as f:
    pickle.dump(class_weights, f)

print('Splits, TF-IDF matrices, and class weights saved.')
print(f'  ML dir : {PROCESSED_ML_DIR}')
print(f'  DL dir : {PROCESSED_DL_DIR}')

Splits, TF-IDF matrices, and class weights saved.
  ML dir : /Users/jia/Documents/TFG/data/processed/ml
  DL dir : /Users/jia/Documents/TFG/data/processed/dl
